# Face-Car: treino do classificador de sonolência (CNN)

Este notebook treina a rede que olha **o rosto do motorista** num quadro da câmera e devolve 3 probabilidades, **nesta ordem**:

| saída | classe | significado |
|---|---|---|
| 0 | `alerta` | motorista acordado |
| 1 | `sonolento` | olhos fechando, cabeça caindo |
| 2 | `bocejo` | bocejando |

**Contrato com o projeto** (o que `entrada/classificador.py` espera):

- entrada: o **quadrado do rosto**, recortado com a regra de `caixa_do_rosto` (em `entrada/classificador.py`) em volta dos pontos do MediaPipe, em **RGB**, redimensionado para `TAMANHO × TAMANHO`, pixels **de 0 a 255** (a normalização é uma camada do modelo). A análise e o simulador recortam igual, porque o `.classes.json` diz `"recorte": "rosto"`;
- saída: `softmax` com 3 valores que somam 1, na ordem acima, que é a mesma de `classificador.classes` em `config/analise.json`.

Junto do `.keras`, o notebook salva `cnn_sonolencia.classes.json` com a ordem das classes. O projeto lê esse arquivo e **se recusa a rodar** se a configuração estiver em outra ordem.

**Dados:** fotos e vídeos em `treinamento/dataset/<classe>/` (seção 1). Dos vídeos são tiradas algumas fotos por segundo. Hoje são os vídeos do **UTA-RLDD** (54 pessoas gravadas pela webcam ou celular, alertas e sonolentas) e as fotos próprias de 2 pessoas, organizados por `treinamento/organizar_dataset.py`.

**Como rodar:** ajuste a célula de **Configuração** e use *Executar tudo*, ou rode `treinamento/treinar.sh` no terminal (veja o README). As células estão na ordem em que precisam rodar.

Abra este notebook no VS Code (ou Jupyter) com o kernel do `.venv` do projeto (`uv sync` instala tudo, inclusive o `ipykernel`). Sem GPU funciona, só que mais devagar. O notebook usa código do projeto: o recorte do rosto (`entrada/`, `treinamento/recortes.py`) e a divisão (`treinamento/divisao.py`). Por isso, e pelo tamanho do dataset (~100 GB), ele não roda mais no Colab.

**Técnica usada (transfer learning):** a rede base (MobileNetV3Large, por padrão) já foi treinada pelo Google com 1,4 milhão de fotos (ImageNet) e sabe extrair bordas, texturas, olhos e rostos. Ela vira a "base" congelada, e só uma camada nova no fim aprende a separar as 3 classes. Depois, a parte final da base é liberada para um **ajuste fino**.

**Resultado:** `modelo/cnn_sonolencia.keras` e `modelo/cnn_sonolencia.classes.json`, na pasta que o projeto usa. O modelo anterior vai para `modelo/anteriores/`.

**Por que o V4.** O primeiro treino deste notebook (o FaceCar_CNN_V3, quadro inteiro) acertou só 13,6% no teste. Ele decorou as pessoas em vez do estado delas, por dois motivos, corrigidos aqui:

1. a pessoa 1 das fotos próprias só aparecia bocejando no treino, e no teste ela caiu alerta e sonolenta. A rede aprendeu "camiseta amarela = bocejo". Agora, quem fica em blocos fica em blocos em **todas** as classes, e a divisão é recusada se uma pessoa aparecer no teste com uma classe que o treino não tem dela (seção 4.5);
2. no quadro inteiro a 224 px, os olhos ocupam poucos pixels, e o que muda de um vídeo do RLDD para outro é roupa, fundo e luz. Agora a rede vê **só o rosto** (seção 4.3).

**Por que o V5.** O V4 (rosto recortado) acertou 49% com as pessoas do RLDD que nunca tinha visto: reconhecia o alerta, mas pegava só 18% dos sonolentos. O RLDD rotula o vídeo inteiro, e a pessoa "sonolenta" passa a maior parte dele de olhos abertos: só 24% dos quadros sonolentos têm o olho mais fechado que o normal dela. A rede aprendia com milhares de quadros "sonolentos" iguais aos alertas. Agora os quadros dos vídeos passam por uma **limpeza pela abertura dos olhos** (seção 4.4), e os vídeos sonolentos rendem mais quadros para compensar (seção 4.2).

**Por que o V6.** No teste ao vivo, o V5 não viu os olhos fechados atrás de óculos e ficou em dúvida sem eles: a webcam, a luz e os óculos eram diferentes dos dados de treino. O V6 acrescenta vídeos gravados nesta webcam com `treinamento/gravar.py` (pasta `gravacoes/` de cada classe), com e sem óculos. Nos vídeos de bocejo gravados, só ficam os quadros com a boca aberta (seção 4.4).

**Por que o V7.** O modelo precisa funcionar com gente que nunca viu. O V5 e o V6 acertavam 98% no treino e ~70% a 78% com pessoas novas: decoravam quem estava no treino. O V7 ataca isso de quatro jeitos (seção 4.5 e 6): **no máximo 60 imagens de cada pessoa em cada classe** no treino (antes, quem tinha 250 quadros sonolentos pesava 4 vezes mais), **ajuste fino mais contido** (20% da base, em vez de 40%), **mais regularização** (Dropout 0,5 e decaimento dos pesos) e **mais aumento de dados** (cor, desfoque, pedaços apagados e qualidade de JPEG). E a nota com 8 pessoas novas variava demais de um treino para outro: `treinamento/validacao_cruzada.py` treina 3 vezes, cada vez com um terço das pessoas no teste, e dá a média.

**O que a validação cruzada mostrou (V8).** Medindo com as 54 pessoas do RLDD (3 dobras de 18), a receita do V6 acertou **80,9% ± 6,5** com gente nova, e a do V7 (teto de 60 imagens por pessoa, ajuste fino em 20%, Dropout 0,5, AdamW e aumento forte) só **75,3% ± 10,3**: o teto tirou do treino justamente os quadros de olho fechando. Por isso o padrão voltou à receita do V6, e as opções do V7 ficaram desligadas. O V8 é essa receita treinada com **todas as pessoas** (`TESTE_SEPARADO = False`, pelo `treinamento/validacao_cruzada.py --final`): sem teste separado, a nota que vale para gente nova é a da validação cruzada, gravada no `.classes.json`.

**Otimizações para acertar mais**, em relação ao FaceCar_CNN_V2 (cada uma é explicada na sua seção):

| o quê | antes (V2) | agora | seção |
|---|---|---|---|
| imagem | quadro inteiro | só o rosto, com o MediaPipe, igual na análise e no simulador | 4.3 |
| rótulo dos quadros de vídeo | o do vídeo inteiro | só quadros em que o olho confirma o rótulo | 4.4 |
| rede base | MobileNetV2 | MobileNetV3Large: mais precisa; ~29 ms por quadro no projeto, contra ~21 ms | 6 |
| resolução | 160 px | 224 px: 2× mais pixels nos olhos e na boca | 1 |
| aumento de dados | flip, rotação ±11°, zoom 5% | + deslocamento, zoom 20%, brilho/contraste, cinza (câmera IR), ruído | 6 |
| alvo do treino | rótulo "duro" | suavização de rótulos (0,1) | 7 |
| escolha da melhor época | erro da validação | F1 macro da validação (todas as classes contam) | 7 |
| taxa de aprendizado | fixa | cai sozinha quando o erro empaca | 7 |
| ajuste fino | — | 40% finais da base, com volta automática se piorar | 8 |
| classes desbalanceadas | — | pesos por classe | 7 |
| dados | fotos, divisão aleatória | fotos e vídeos, divisão por pessoa; nota separada para as pessoas que a rede nunca viu | 4 e 9 |

## 1. Configuração

Tudo o que muda de um treino para outro fica aqui.

O dataset fica em **`treinamento/dataset`**, com uma subpasta por classe. Dentro de cada uma pode haver **fotos e vídeos** misturados, inclusive em subpastas. **O nome de cada arquivo começa pela pessoa**, seguida de `_`:

```
treinamento/dataset/
├── alerta/      ← fotos (.jpg .png …) e/ou vídeos (.mp4 .mov …)
│   ├── rldd/      rldd01_0.mov, rldd02_0.mov …        (UTA-RLDD, vídeo "0")
│   ├── proprio/   proprio1_acordado_0001.jpg …
│   └── gravacoes/ voce1_oculos_alerta.mp4 …            (treinamento/gravar.py, nesta webcam)
├── sonolento/
│   ├── rldd/      rldd01_10.MOV …                      (UTA-RLDD, vídeo "10")
│   └── proprio/   proprio1_sonolento_0001.jpg, proprio1_olhos_fechados_0002.jpg …
├── bocejo/
│   └── proprio/   proprio1_bocejo_0004.jpg …
└── _fora_do_treino/   ← não está em CLASSES, então o treino ignora
    ├── pouco_alerta/  vídeos "5" do RLDD: num quadro isolado, iguais ao alerta
    └── duvidosas/     fotos cujo rosto contradiz a pasta (bocejo de boca fechada, piscada…)
```

Para acrescentar dados, ponha o arquivo na pasta da classe com o nome `pessoa_qualquercoisa` (ex.: `maria_bocejo_01.mp4`). Sempre o mesmo nome para a mesma pessoa.

- `CLASSES` liga **cada saída do modelo** à **subpasta** que tem os exemplos dela. A **ordem das chaves** é a ordem das saídas. Se a pasta tiver outro nome, mude só o valor, nunca a ordem.
- `QUADROS_POR_SEGUNDO` e `MAX_QUADROS_POR_VIDEO` dizem quantas fotos são tiradas de cada vídeo (seção 4).
- `BASE` e `TAMANHO` escolhem a rede base e a resolução (seção 6).
- `PROPORCOES` divide os dados em **treino** (a rede aprende), **validação** (decide quando parar) e **teste** (nota final, usado uma vez só, no fim).
- `REGEX_PESSOA`: veja a seção 4.

In [ ]:
# Saída do modelo → subpasta do dataset. A ORDEM é a das saídas e tem que ser a de
# classificador.classes em config/analise.json: alerta, sonolento, bocejo.
CLASSES = {
    "alerta": "alerta",
    "sonolento": "sonolento",
    "bocejo": "bocejo",
}

# None = automático: treinamento/dataset, treinamento/resultados (gráficos) e modelo/ na raiz do
# projeto (o .keras final).
PASTA_DATASET = None
PASTA_SAIDA = None
PASTA_MODELO = None

NOME_MODELO = "FaceCar_CNN_V8"
ARQUIVO_MODELO = "cnn_sonolencia"   # gera cnn_sonolencia.keras e cnn_sonolencia.classes.json

QUADROS_POR_SEGUNDO = 2      # fotos tiradas de cada segundo de vídeo
# Teto de quadros por vídeo, por classe (None = sem teto). Os vídeos do RLDD têm ~10 min: 60 quadros
# = 1 a cada 10 s. O sonolento tira 300 (1 a cada 2 s) porque a limpeza da seção 4.4 deixa só ~1/4.
MAX_QUADROS_POR_VIDEO = {"alerta": 60, "sonolento": 300, "bocejo": 60}

LIMPAR_ROTULOS = True   # quadros de vídeo só ficam se o olho confirmar o rótulo (seção 4.4)
FRACAO_OLHO = 0.75      # "olho fechando" = EAR abaixo de 75% da abertura normal da pessoa

RECORTAR_ROSTO = True       # a rede vê só o rosto (seção 4.3); False = quadro inteiro, como o V3

BASE = "MobileNetV3Large"   # ou "EfficientNetV2B0" (mais precisa, ~2x mais lenta) ou "MobileNetV2"
TAMANHO = 224                     # lado da imagem de entrada (o FaceCar_CNN_V2 usava 160)
LOTE = 32                         # imagens por passo de treino
PROPORCOES = (0.70, 0.15, 0.15)   # treino, validação, teste
SEED = 42                         # mesma semente → mesma divisão e mesmo sorteio

EPOCAS = 25                  # máximo da fase 1; o EarlyStopping costuma parar antes
AJUSTE_FINO = True           # fase 2: libera a parte final da rede base
EPOCAS_AJUSTE_FINO = 25
FRACAO_LIBERADA = 0.4        # fração final das camadas da base que volta a aprender na fase 2
SUAVIZACAO_ROTULOS = 0.1     # seção 7
PESOS_BASE = "imagenet"      # pesos pré-treinados da rede base

# Separação por pessoa (seção 4). Grupo 1 da regex = quem aparece, tirado do NOME do arquivo
# (da foto ou do vídeo): o que vem antes do primeiro "_". "rldd07_10.mp4" → "rldd07",
# "proprio1_bocejo_0004.jpg" → "proprio1". None = separa por vídeo/foto.
REGEX_PESSOA = r"^([^_]+)_"
FOTOS_POR_BLOCO = 15   # pessoas de uma classe com pessoas de menos (o bocejo): seção 4.5

# Opções do V7 contra a decoreba das pessoas (seções 4.5 e 6). Na validação cruzada, ligadas,
# pioraram: 75,3% com gente nova, contra 80,9% da receita do V6. Ficam desligadas. Para ligar:
# MAX_POR_PESSOA_CLASSE = {"alerta": 60, "sonolento": 60}, FRACAO_LIBERADA = 0.2, DROPOUT = 0.5,
# DECAIMENTO_PESOS = 1e-4 e AUMENTO_FORTE = True.
MAX_POR_PESSOA_CLASSE = None # imagens de cada pessoa por classe no treino (None = sem teto)
DROPOUT = 0.3                # fração das entradas da última camada desligadas a cada passo
DECAIMENTO_PESOS = 0.0       # AdamW: puxa os pesos para zero (0 = Adam comum)
AUMENTO_FORTE = False        # cor, desfoque, pedaços apagados e qualidade de JPEG (seção 6)

# Validação cruzada por pessoa (treinamento/validacao_cruzada.py muda estes valores): com DOBRA
# = k, o teste é o k-ésimo terço das pessoas e o modelo não vai para modelo/.
DOBRAS = 3
DOBRA = None
SALVAR_MODELO = True
# Treino final (validacao_cruzada.py --final): todas as pessoas no treino e na validação, sem teste
# separado; a nota com gente nova é a da validação cruzada, lida deste resumo.
TESTE_SEPARADO = True
RESUMO_VALIDACAO_CRUZADA = None

CACHE_NA_MEMORIA = True      # guarda as imagens já decodificadas na RAM (desligue se faltar memória)

## 2. Bibliotecas

Só o que é usado: **TensorFlow/Keras** (rede e leitura das imagens), **OpenCV** (quadros dos vídeos), **MediaPipe** (rosto, pelo código do projeto), **scikit-learn** (divisão dos dados e métricas), **NumPy** e **matplotlib**.

O código do projeto é importado da raiz do repositório, que a célula acha subindo a partir da pasta atual.

`keras.utils.set_random_seed` fixa de uma vez as sementes do Python, do NumPy e do TensorFlow. A GPU ainda pode dar diferenças minúsculas entre execuções, mas a divisão dos dados sai sempre igual.

In [ ]:
import json
import os
import re
import shutil
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import date
from pathlib import Path

import cv2
import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from keras import layers
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

RAIZ_PROJETO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "entrada").is_dir())
sys.path.insert(0, str(RAIZ_PROJETO))
from entrada.classificador import FATOR_RECORTE
from entrada.contexto import carregar_config
from treinamento import recortes
from treinamento.divisao import (agrupar, limitar_por_pessoa, mover_para_o_treino,
                                 pessoas_com_classe_que_o_treino_nao_viu)
from treinamento.rotulos import limpar

keras.utils.set_random_seed(SEED)
NOMES = list(CLASSES)   # ["alerta", "sonolento", "bocejo"]: índice = saída do modelo

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| OpenCV", cv2.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "nenhuma (o treino vai ser lento)")
print("Projeto:", RAIZ_PROJETO)

## 3. Onde estão os dados

Usa `treinamento/dataset` direto, põe gráficos em `treinamento/resultados/` e o modelo final em `modelo/`, na raiz do projeto. Os quadros tirados dos vídeos ficam em `treinamento/quadros/` (seção 4.2) e os rostos recortados em `treinamento/rostos/` (seção 4.3). Tudo isso fica fora do git.

In [ ]:
PASTA_DATASET = PASTA_DATASET or str(RAIZ_PROJETO / "treinamento" / "dataset")
PASTA_SAIDA = PASTA_SAIDA or str(RAIZ_PROJETO / "treinamento" / "resultados")
PASTA_MODELO = PASTA_MODELO or str(RAIZ_PROJETO / "modelo")

PASTA_QUADROS = Path(PASTA_DATASET).parent / "quadros"
PASTA_ROSTOS = Path(PASTA_DATASET).parent / "rostos" / recortes.versao()
Path(PASTA_SAIDA).mkdir(parents=True, exist_ok=True)
Path(PASTA_MODELO).mkdir(parents=True, exist_ok=True)
print("Dataset:   ", PASTA_DATASET)
print("Quadros:   ", PASTA_QUADROS)
print("Rostos:    ", PASTA_ROSTOS)
print("Resultados:", PASTA_SAIDA)
print("Modelo:    ", PASTA_MODELO)

## 4. Fotos, vídeos e a divisão em treino, validação e teste

### 4.1 O que há em cada classe

A célula confere se as subpastas de `CLASSES` existem e conta fotos e vídeos. Arquivos de outro tipo são listados e ignorados, nunca descartados em silêncio. As fotos precisam estar em JPG, PNG ou BMP, os formatos que o TensorFlow lê. Se houver `.webp` ou `.heic`, converta antes.

In [ ]:
FOTOS = {".jpg", ".jpeg", ".png", ".bmp"}
VIDEOS = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v", ".3gp"}

raiz = Path(PASTA_DATASET)
if not raiz.is_dir():
    raise FileNotFoundError(f"A pasta do dataset não existe: {raiz}")
subpastas = sorted(p.name for p in raiz.iterdir() if p.is_dir())
faltando = [pasta for pasta in CLASSES.values() if pasta not in subpastas]
if faltando:
    raise FileNotFoundError(f"Subpastas {faltando} não existem em {raiz}. Existem: {subpastas}. "
                            "Crie-as ou ajuste CLASSES na configuração.")
ignoradas = sorted(set(subpastas) - set(CLASSES.values()))
if ignoradas:
    print("Subpastas ignoradas (não estão em CLASSES):", ignoradas)

fotos, videos, outros = {}, {}, []
for nome, pasta in CLASSES.items():
    arquivos = sorted(f for f in (raiz / pasta).rglob("*") if f.is_file() and not f.name.startswith("."))
    fotos[nome] = [f for f in arquivos if f.suffix.lower() in FOTOS]
    videos[nome] = [f for f in arquivos if f.suffix.lower() in VIDEOS]
    outros += [f for f in arquivos if f.suffix.lower() not in FOTOS | VIDEOS]
    print(f"{nome:<10} pasta '{pasta}': {len(fotos[nome])} fotos, {len(videos[nome])} vídeos")
if outros:
    print(f"\nIgnorados {len(outros)} arquivos de tipo não suportado, por exemplo:",
          [str(f.relative_to(raiz)) for f in outros[:5]])

### 4.2 Quadros dos vídeos

A rede aprende com **fotos**, então cada vídeo vira uma sequência delas. Não se usa todo quadro: um vídeo a 30 quadros por segundo tem 30 fotos quase iguais a cada segundo. Isso só deixa o treino mais lento e faz o vídeo pesar demais. A célula tira `QUADROS_POR_SEGUNDO` fotos por segundo, espaçadas por igual, e no máximo `MAX_QUADROS_POR_VIDEO` por vídeo. O teto é por classe: os vídeos sonolentos rendem 300 quadros, porque a limpeza da seção 4.4 descarta uns 3/4 deles.

- Os quadros são salvos em `quadros/<classe>/<vídeo>@<configuração>/`, com o lado maior reduzido a 640 px (o rosto é recortado deles depois, na seção 4.3).
- A extração acontece **uma vez só**. Na próxima execução, os quadros prontos são reaproveitados. Se `QUADROS_POR_SEGUNDO` ou o teto mudarem, o nome da pasta muda e os quadros são extraídos de novo. Uma extração interrompida também é refeita.
- O OpenCV entrega os quadros já desvirados, mesmo de vídeo de celular gravado em pé.
- Vários vídeos são processados ao mesmo tempo, um por núcleo do processador.

In [ ]:
LADO_MAXIMO = 640


def teto(nome):
    return MAX_QUADROS_POR_VIDEO.get(nome) if isinstance(MAX_QUADROS_POR_VIDEO, dict) else MAX_QUADROS_POR_VIDEO


def config_quadros(nome):
    return f"{QUADROS_POR_SEGUNDO}qps-max{teto(nome) or 'livre'}"


def pasta_dos_quadros(video, nome):
    relativo = video.relative_to(raiz / CLASSES[nome])
    return PASTA_QUADROS / CLASSES[nome] / f"{'__'.join(relativo.parts)}@{config_quadros(nome)}"


def extrair_quadros(video, destino, maximo):
    """Salva os quadros escolhidos do vídeo em `destino`; devolve os arquivos, em ordem."""
    marca = destino / ".completo"
    if marca.exists():
        return sorted(destino.glob("q*.jpg"))
    shutil.rmtree(destino, ignore_errors=True)
    destino.mkdir(parents=True)

    captura = cv2.VideoCapture(str(video))
    if not captura.isOpened():
        print(f"  Aviso: não consegui abrir {video.name}; vídeo ignorado")
        return []
    fps = captura.get(cv2.CAP_PROP_FPS) or 30
    total = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))
    passo = max(1, round(fps / QUADROS_POR_SEGUNDO))
    escolhidos = np.arange(0, max(total, 1), passo)
    if maximo and len(escolhidos) > maximo:
        escolhidos = escolhidos[np.linspace(0, len(escolhidos) - 1, maximo).astype(int)]
    escolhidos = set(escolhidos.tolist()) if total > 0 else None   # sem total: só o passo

    salvos, indice = [], 0
    while captura.grab():   # grab() só avança; retrieve() decodifica, e só dos escolhidos
        if (indice % passo == 0) if escolhidos is None else (indice in escolhidos):
            ok, quadro = captura.retrieve()
            if ok:
                escala = LADO_MAXIMO / max(quadro.shape[:2])
                if escala < 1:
                    quadro = cv2.resize(quadro, None, fx=escala, fy=escala,
                                        interpolation=cv2.INTER_AREA)
                arquivo = destino / f"q{indice:07d}.jpg"
                cv2.imwrite(str(arquivo), quadro, [cv2.IMWRITE_JPEG_QUALITY, 95])
                salvos.append(arquivo)
        indice += 1
        if maximo and len(salvos) >= maximo \
                or escolhidos and indice > max(escolhidos):
            break
    captura.release()
    marca.write_text(json.dumps({"video": str(video), "fps": fps, "quadros": len(salvos)}))
    return salvos


tarefas = [(nome, video) for nome in NOMES for video in videos[nome]]
inicio = time.time()
with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
    quadros = list(executor.map(lambda t: extrair_quadros(t[1], pasta_dos_quadros(t[1], t[0]),
                                                          teto(t[0])), tarefas))
quadros_do_video = {video: arquivos for (_, video), arquivos in zip(tarefas, quadros)}
if tarefas:
    print(f"{len(tarefas)} vídeos → {sum(map(len, quadros))} quadros "
          f"em {time.time() - inicio:.0f} s")
    for nome in NOMES:
        if videos[nome]:
            print(f"  {nome:<10} {sum(len(quadros_do_video[v]) for v in videos[nome]):>6} quadros "
                  f"({config_quadros(nome)})")

In [ ]:
# Junta fotos e quadros numa lista só, com a classe e a pessoa de cada imagem.
def pessoa(arquivo):
    achou = re.search(REGEX_PESSOA, Path(arquivo).name)
    if not achou:
        raise ValueError(f"REGEX_PESSOA não achou a pessoa em '{Path(arquivo).name}'")
    return achou.group(1)


caminhos, rotulos, pessoas, de_video = [], [], [], []
for indice, nome in enumerate(NOMES):
    origens = [(foto, [foto], False) for foto in fotos[nome]]
    origens += [(video, quadros_do_video[video], True) for video in videos[nome]]
    for origem, arquivos, eh_video in origens:
        caminhos += [str(a) for a in arquivos]
        rotulos += [indice] * len(arquivos)
        pessoas += [pessoa(origem) if REGEX_PESSOA else str(origem)] * len(arquivos)
        de_video += [eh_video] * len(arquivos)
caminhos, rotulos = np.array(caminhos), np.array(rotulos)
pessoas, de_video = np.array(pessoas, dtype=object), np.array(de_video)
print(f"{len(caminhos)} imagens: {(~de_video).sum()} fotos e {de_video.sum()} quadros de vídeo")

### 4.3 Recorte do rosto

A rede vê **só o rosto**. Cada foto e cada quadro passa pelo **MediaPipe Face Landmarker**, o mesmo detector da análise em tempo real, com as opções de `config/analise.json`. Do maior rosto (o do motorista, se houver alguém ao fundo) sai um **quadrado** centrado nos pontos do rosto, com lado igual ao maior lado da caixa dos pontos × `FATOR_RECORTE` (1,4). Isso sobra testa, queixo e um pouco de fundo, para a cabeça inclinada caber.

- A regra é a função `caixa_do_rosto` de `entrada/classificador.py`. A análise (`python -m entrada`) e o simulador (`simulador/js/recorte.js`) usam a mesma, e `testes/casos/recorte.json` confere que as três dão o mesmo quadrado. O modelo recebe no uso a mesma imagem do treino.
- Por que recortar: no quadro inteiro reduzido a 224 px, os olhos ocupam poucos pixels, e o que mais muda de um vídeo para outro é roupa, fundo e luz. Foi isso que o V3 decorou.
- Imagem **sem rosto** fica fora do treino. Na análise, sem rosto o modelo nem é chamado.
- Junto com o recorte, ficam as **medidas do rosto** (abertura dos olhos, da boca e inclinação da cabeça, com as fórmulas da análise), usadas pela limpeza da seção 4.4.
- Os recortes ficam em `treinamento/rostos/<versão>/`, espelhando o caminho original, e são feitos **uma vez só**. Se a regra mudar, a versão muda e os recortes são refeitos. Vários processos trabalham ao mesmo tempo.

In [ ]:
ear = np.full(len(caminhos), np.nan)
boca = np.full(len(caminhos), np.nan)
if RECORTAR_ROSTO or LIMPAR_ROTULOS:   # o recorte também mede o rosto
    inicio = time.time()
    raiz_treinamento = Path(PASTA_DATASET).parent   # contém dataset/ e quadros/
    destinos = [recortes.destino(c, raiz_treinamento, PASTA_ROSTOS) for c in caminhos]
    tem_rosto = np.array(recortes.recortar(caminhos, destinos))
    print(f"Rostos recortados em {time.time() - inicio:.0f} s (fator {FATOR_RECORTE})")
    print(recortes.resumo_sem_rosto(tem_rosto, rotulos, NOMES))
    medidas = [recortes.ler_medidas(d) for d in destinos]
    ear = np.array([m["ear"] if m else np.nan for m in medidas])
    boca = np.array([m["boca"] if m else np.nan for m in medidas])
    if RECORTAR_ROSTO:
        caminhos = np.array([str(d) for d in destinos])
    caminhos, rotulos, pessoas = caminhos[tem_rosto], rotulos[tem_rosto], pessoas[tem_rosto]
    de_video, ear, boca = de_video[tem_rosto], ear[tem_rosto], boca[tem_rosto]

    amostra = np.random.default_rng(SEED).choice(len(caminhos), size=min(12, len(caminhos)), replace=False)
    plt.figure(figsize=(12, 3.4))
    for k, i in enumerate(amostra):
        plt.subplot(2, 6, k + 1)
        plt.imshow(cv2.cvtColor(cv2.imread(caminhos[i]), cv2.COLOR_BGR2RGB))
        plt.title(f"{NOMES[rotulos[i]]}\n{pessoas[i]}", fontsize=8)
        plt.axis("off")
    plt.suptitle("Rostos recortados (confira se o rosto está inteiro no quadrado)", y=1.06)
    plt.show()

### 4.4 Limpeza dos rótulos dos vídeos

O UTA-RLDD dá um rótulo ao **vídeo inteiro**. Mas a pessoa do vídeo "sonolento" passa a maior parte dele de olhos abertos, com cara de alerta: medindo, só ~24% dos quadros sonolentos têm o olho mais fechado que o normal dela. Ensinar à rede que esses quadros são "sonolento" é ensinar errado, e foi o que fez o V4 pegar só 18% dos sonolentos de gente nova.

A regra (`treinamento/rotulos.py`, com testes em `testes/treinamento/test_rotulos.py`) usa a **abertura dos olhos** (EAR) de cada quadro:

- a **abertura normal** de cada pessoa é a mediana do EAR nos quadros do vídeo alerta dela. É por pessoa porque o tamanho do olho varia muito;
- quadro de vídeo **sonolento** só fica se o olho estiver abaixo de `FRACAO_OLHO` (75%) dessa abertura: olho fechando;
- quadro de vídeo **alerta** sai se estiver abaixo desse corte: são as piscadas;
- quadro de vídeo de **bocejo** (só há nas gravações guiadas) fica só com a boca aberta (`rosto.abertura_boca_aberta` de `config/analise.json`): entre um bocejo e outro, a boca fica fechada;
- **fotos** não mudam: as fotos próprias já foram conferidas uma a uma.

A inclinação da cabeça não entra: a câmera muda de lugar entre os dois vídeos da mesma pessoa, então o ângulo de um não se compara com o do outro.

**Consequência para a nota:** validação e teste também são limpos. A nota mede se a rede separa "olho fechando" de "olho aberto" em gente nova, que é o que um quadro sozinho consegue mostrar. Ela não mede se a rede adivinha o vídeo de que o quadro veio.

In [ ]:
if LIMPAR_ROTULOS:
    manter, referencia = limpar(rotulos, pessoas, de_video, ear, NOMES, FRACAO_OLHO, boca=boca,
                                boca_aberta=carregar_config()["rosto"]["abertura_boca_aberta"])
    print(f"Abertura normal dos olhos de {len(referencia)} pessoas (mediana do EAR no vídeo alerta)")
    print(f"{'quadros de vídeo':<18}{'antes':>7}{'ficam':>7}")
    for indice, nome in enumerate(NOMES):
        dela = de_video & (rotulos == indice)
        if dela.any():
            print(f"{nome:<18}{dela.sum():>7}{(dela & manter).sum():>7}")
    sonolentas = set(pessoas[de_video & (rotulos == NOMES.index("sonolento"))])
    sem_quadro = sorted(sonolentas - set(pessoas[de_video & manter & (rotulos == NOMES.index("sonolento"))]))
    print(f"\n{len(sem_quadro)} de {len(sonolentas)} pessoas ficaram sem quadro sonolento (o olho "
          f"não fechou no vídeo): {', '.join(sem_quadro) or '-'}")
    caminhos, rotulos, pessoas = caminhos[manter], rotulos[manter], pessoas[manter]
    de_video, ear, boca = de_video[manter], ear[manter], boca[manter]

### 4.5 Grupos e divisão

Cada exemplo (foto ou quadro de vídeo) recebe um **grupo**, que diz de onde ele veio:

- quadro de vídeo → o **vídeo**;
- foto → a própria foto;
- com `REGEX_PESSOA` (o padrão) → a **pessoa**, tirada do nome da foto ou do vídeo.

**Um grupo nunca é dividido**: todos os quadros de um vídeo caem no mesmo conjunto. Isso evita o **vazamento de dados**. Dois quadros vizinhos de um vídeo são quase a mesma foto. Se um caísse no treino e o outro no teste, a rede acertaria o teste só por "lembrar" do vídeo, e a nota ficaria perto de 100% sem valer para vídeos novos. Pelo mesmo motivo, dividir **por pessoa** é ainda melhor, porque mede se a rede funciona com **gente que ela nunca viu**.

A divisão é **estratificada por grupo** (`StratifiedGroupKFold`): respeita os grupos e, dentro disso, tenta manter em cada conjunto a mesma proporção de alerta, sonolento e bocejo. Com poucos vídeos ou pessoas, as proporções ficam aproximadas. Por isso, grave **vários vídeos curtos por classe** em vez de um longo: cada classe precisa de pelo menos uns 7 grupos para aparecer nos três conjuntos.

**Classe com pessoas de menos.** Para separar 15% das pessoas, uma classe precisa de pelo menos 7 pessoas. Senão, algum conjunto fica sem ela, e a validação não consegue medir essa classe. Hoje é o caso do bocejo, que só tem fotos de 2 pessoas (as das fotos próprias). As fotos **dessas pessoas**, em **todas as classes delas**, são repartidas em blocos de `FOTOS_POR_BLOCO` fotos seguidas, e cada bloco é um grupo. Assim cada uma aparece alerta, sonolenta e bocejando no treino, e a rede não pode acertar a classe pelo jeito da pessoa. As pessoas do RLDD continuam divididas por pessoa. Com bocejos de 7 pessoas ou mais, os blocos deixam de existir sozinhos.

Os blocos têm um preço: a mesma pessoa fica no treino e no teste, então **a nota dessas fotos no teste é otimista**. Por isso a seção 9 mostra, à parte, a nota das **pessoas que a rede nunca viu**, que é a que vale para um motorista novo.

**Teto por pessoa.** No treino, cada pessoa entra com no máximo `MAX_POR_PESSOA_CLASSE` imagens de cada classe (60 de alerta e 60 de sonolento), espaçadas ao longo do vídeo. Quem tinha 250 quadros sonolentos pesava 4 vezes mais que quem tinha 60, e a rede aprendia o rosto dessa pessoa. Validação e teste não têm teto.

**Validação cruzada.** Com `DOBRA` definido (pelo `treinamento/validacao_cruzada.py`), o teste deixa de ser um sorteio de 15% e passa a ser um terço das pessoas: em 3 treinos, cada pessoa cai no teste uma vez, e a média das 3 notas varia bem menos que a nota de um teste com 8 pessoas.

**Conferência.** Foi o que derrubou o V3: só o bocejo virou blocos, e a pessoa 1 apareceu no treino só bocejando e no teste alerta e sonolenta. Depois da divisão, a célula **devolve ao treino** os blocos de quem está no treino e apareceria na validação ou no teste com uma classe que o treino não tem dela (`mover_para_o_treino`), e depois confere (`pessoas_com_classe_que_o_treino_nao_viu`, em `treinamento/divisao.py`, com testes em `testes/treinamento/test_divisao.py`).

Se uma classe tiver muito mais exemplos que outra, a rede tende a "chutar" a maior. Na seção 7, cada classe ganha um **peso** inverso ao seu tamanho para compensar.

In [ ]:
# Pessoas de uma classe com pessoas de menos para a divisão por pessoa: as fotos delas, em todas
# as classes, viram blocos de fotos seguidas (treinamento/divisao.py).
_, fracao_val, fracao_teste = PROPORCOES
GRUPOS_MINIMOS = max(round(1 / fracao_teste), round((1 - fracao_teste) / fracao_val))
if REGEX_PESSOA:
    grupos, pessoas_em_blocos = agrupar(caminhos, rotulos, pessoas, NOMES, GRUPOS_MINIMOS,
                                        FOTOS_POR_BLOCO)
else:
    grupos, pessoas_em_blocos = pessoas.copy(), []
em_bloco = np.array(["/bloco" in g for g in grupos])
if pessoas_em_blocos:
    poucas = [n for i, n in enumerate(NOMES) if len(set(pessoas[rotulos == i])) < GRUPOS_MINIMOS]
    print(f"Atenção: {poucas} têm menos de {GRUPOS_MINIMOS} pessoas, o mínimo para dividir por "
          f"pessoa. As fotos de {', '.join(pessoas_em_blocos)}, em todas as classes delas, foram "
          f"repartidas em blocos de {FOTOS_POR_BLOCO}: a nota delas no teste fica otimista, e a "
          "seção 9 mostra à parte a das pessoas que a rede nunca viu.\n")

contagem = np.bincount(rotulos, minlength=len(NOMES))
print(f"{'':<10}{'fotos':>7}{'quadros':>9}{'total':>7}{'grupos':>8}")
for indice, nome in enumerate(NOMES):
    da_classe = rotulos == indice
    print(f"{nome:<10}{(da_classe & ~de_video).sum():>7}{(da_classe & de_video).sum():>9}"
          f"{da_classe.sum():>7}{len(set(grupos[da_classe])):>8}")
if contagem.min() == 0:
    vazias = [NOMES[i] for i in np.flatnonzero(contagem == 0)]
    raise ValueError(f"Classe sem nenhum exemplo: {vazias}. Coloque fotos ou vídeos na pasta dela.")
if contagem.max() / contagem.min() > 1.5:
    print(f"\nAtenção: classes desbalanceadas (a maior tem {contagem.max() / contagem.min():.1f}x "
          "a menor). Os pesos por classe da seção 7 compensam.")

In [ ]:
_, FRACAO_VAL, FRACAO_TESTE = PROPORCOES


def separar(indices, fracao):
    """Separa ~`fracao` dos grupos de `indices`, estratificando pela classe."""
    partes = max(2, round(1 / fracao))
    if len(set(grupos[indices])) < partes:
        raise ValueError(f"Só há {len(set(grupos[indices]))} grupos (vídeos/fotos/pessoas) para "
                         f"separar {fracao:.0%}; são precisos pelo menos {partes}. Adicione dados "
                         "ou mude PROPORCOES.")
    corte = StratifiedGroupKFold(n_splits=partes, shuffle=True, random_state=SEED)
    fica, sai = next(corte.split(indices, rotulos[indices], grupos[indices]))
    return indices[fica], indices[sai]


if DOBRA is None and TESTE_SEPARADO:
    resto, idx_teste = separar(np.arange(len(caminhos)), FRACAO_TESTE)
    idx_treino, idx_val = separar(resto, FRACAO_VAL / (1 - FRACAO_TESTE))
elif DOBRA is None:   # treino final: todas as pessoas; só a validação, para parar e escolher a época
    idx_treino, idx_val = separar(np.arange(len(caminhos)), FRACAO_VAL)
    idx_teste = None
    print("Treino final, sem teste separado: a nota com gente nova é a da validação cruzada.\n")
else:   # validação cruzada: o teste é a dobra DOBRA das DOBRAS
    corte = StratifiedGroupKFold(n_splits=DOBRAS, shuffle=True, random_state=SEED)
    resto, idx_teste = list(corte.split(caminhos, rotulos, grupos))[DOBRA]
    idx_treino, idx_val = separar(resto, FRACAO_VAL / (1 - 1 / DOBRAS))
    print(f"Validação cruzada: dobra {DOBRA + 1} de {DOBRAS}\n")

conjuntos = {"treino": idx_treino, "validação": idx_val}
if idx_teste is not None:
    conjuntos["teste"] = idx_teste
nomes_conj = list(conjuntos)
for i, a in enumerate(nomes_conj):
    for b in nomes_conj[i + 1:]:
        assert not set(grupos[conjuntos[a]]) & set(grupos[conjuntos[b]]), f"um grupo caiu em {a} e em {b}"
# Pessoa em blocos que ficaria sem uma classe no treino: esses blocos voltam para o treino.
conjuntos, movidos = mover_para_o_treino(conjuntos, pessoas, rotulos, NOMES)
for movido in movidos:
    print("Movido para o treino:", movido)
problemas = pessoas_com_classe_que_o_treino_nao_viu(conjuntos, pessoas, rotulos, NOMES)
if problemas:
    raise ValueError("A rede poderia acertar a classe pelo jeito da pessoa (foi o erro do V3):\n  "
                     + "\n  ".join(problemas))
# Teto por pessoa e classe, só no treino: ninguém pesa mais que os outros.
antes = len(conjuntos["treino"])
conjuntos["treino"] = limitar_por_pessoa(conjuntos["treino"], pessoas, rotulos, caminhos, NOMES,
                                         MAX_POR_PESSOA_CLASSE)
if len(conjuntos["treino"]) < antes:
    print(f"Teto por pessoa e classe {MAX_POR_PESSOA_CLASSE}: o treino foi de {antes} para "
          f"{len(conjuntos['treino'])} imagens\n")
idx_treino, idx_val = conjuntos["treino"], conjuntos["validação"]
# sem teste separado, as seções 9 e 10 usam a validação (otimista: ela escolheu a melhor época)
idx_teste = conjuntos.get("teste", idx_val)

print(f"{'':<10}" + "".join(f"{n:>11}" for n in NOMES) + f"{'total':>8}{'grupos':>8}"
      f"{'pessoas novas':>15}")
for nome, idx in conjuntos.items():
    por_classe = np.bincount(rotulos[idx], minlength=len(NOMES))
    novas = sorted(set(pessoas[idx][~em_bloco[idx]])) if nome != "treino" else []
    print(f"{nome:<10}" + "".join(f"{n:>11}" for n in por_classe)
          + f"{len(idx):>8}{len(set(grupos[idx])):>8}{len(novas):>15}")
    if por_classe.min() == 0:
        print(f"  Atenção: '{nome}' ficou sem alguma classe; adicione mais vídeos/fotos dela.")

## 5. Leitura das imagens (`tf.data`)

Cada imagem passa por estes passos:

1. lê o arquivo e decodifica em **RGB** (3 canais, mesmo se a foto for cinza);
2. redimensiona o quadrado do rosto (ou o quadro inteiro, com `RECORTAR_ROSTO = False`) para `TAMANHO × TAMANHO`, como `preparar()` faz no projeto com `cv2.resize`. Treino e uso precisam preparar a imagem do mesmo jeito;
3. arredonda para inteiros de 0 a 255, como a câmera entrega. Assim o cache ocupa 4× menos memória;
4. `cache`: depois da 1ª época, as imagens já vêm prontas da memória;
5. só no treino: `shuffle` embaralha a cada época, e um **chuvisco** leve (`RUIDO`) imita o ruído de webcam em pouca luz. Ele é sorteado de novo a cada época;
6. o rótulo vira **one-hot**: `sonolento` = `[0, 1, 0]`. É o formato que a suavização de rótulos e a métrica F1 da seção 7 precisam;
7. `batch` junta em lotes de `LOTE`, e `prefetch` prepara o próximo lote enquanto a GPU treina o atual.

Arquivo corrompido não derruba o treino: é pulado com um aviso (`ignore_errors`).

In [ ]:
def carregar_imagem(caminho, rotulo):
    imagem = tf.io.decode_image(tf.io.read_file(caminho), channels=3, expand_animations=False)
    imagem = tf.image.resize(imagem, (TAMANHO, TAMANHO))   # bilinear; o rosto já é quadrado
    imagem = tf.cast(tf.clip_by_value(tf.round(imagem), 0, 255), tf.uint8)
    return imagem, rotulo


def para_o_modelo(imagem, rotulo):
    # o modelo recebe float32 de 0 a 255; o rótulo vai em one-hot
    return tf.cast(imagem, tf.float32), tf.one_hot(rotulo, len(NOMES))


def com_ruido(imagem, rotulo):
    # chuvisco de webcam em pouca luz: desvio de RUIDO níveis (de 0 a 255), sorteado a cada época
    ruido = tf.random.normal(tf.shape(imagem), stddev=RUIDO)
    return tf.clip_by_value(imagem + ruido, 0, 255), rotulo


def com_jpeg(imagem, rotulo):
    # qualidade de JPEG sorteada a cada época: a rede não pode depender de como a imagem foi
    # comprimida (com o V5, o mesmo JPEG lido pelo OpenCV e pelo TensorFlow dava 5 pontos de diferença)
    imagem = tf.image.random_jpeg_quality(imagem, 50, 95)
    return tf.ensure_shape(imagem, (TAMANHO, TAMANHO, 3)), rotulo   # o JPEG perde o tamanho fixo


def montar(indices, treinar):
    ds = tf.data.Dataset.from_tensor_slices((caminhos[indices], rotulos[indices]))
    ds = ds.map(carregar_imagem, num_parallel_calls=tf.data.AUTOTUNE).ignore_errors(log_warning=True)
    if CACHE_NA_MEMORIA:
        ds = ds.cache()
    if treinar:
        ds = ds.shuffle(len(indices), seed=SEED, reshuffle_each_iteration=True)
    if treinar and AUMENTO_FORTE:
        ds = ds.map(com_jpeg, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(para_o_modelo, num_parallel_calls=tf.data.AUTOTUNE)
    if treinar and RUIDO:
        ds = ds.map(com_ruido, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(LOTE).prefetch(tf.data.AUTOTUNE)


RUIDO = 6.0
treino = montar(idx_treino, treinar=True)
validacao = montar(idx_val, treinar=False)
teste = montar(idx_teste, treinar=False)

imagens, rots = next(iter(treino))
plt.figure(figsize=(9, 9))
for i in range(min(9, len(imagens))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(imagens[i].numpy().astype("uint8"))
    plt.title(NOMES[int(np.argmax(rots[i]))])
    plt.axis("off")
plt.suptitle("Amostras do treino (confira se o rótulo bate com a foto)")
plt.show()

## 6. O modelo

Uma imagem atravessa estas camadas (formas para a MobileNetV3Large a 224 px):

| camada | o que faz | forma |
|---|---|---|
| `imagem_motorista` | entrada RGB, pixels de 0 a 255 | 224×224×3 |
| `aumento_dados` | **só no treino**: cria variações da foto (abaixo) | 224×224×3 |
| *base* (`BASE`) | normaliza os pixels e extrai características visuais | 7×7×960 |
| `GlobalAveragePooling2D` | média de cada mapa de características: resume a imagem em um vetor | 960 |
| `Dropout(DROPOUT)` | **só no treino**: desliga 50% das entradas ao acaso, para não decorar | 960 |
| `classificacao` | `Dense` + `softmax`: 3 probabilidades que somam 1 | 3 |

**Rede base.** A padrão é a **MobileNetV3Large**, sucessora da MobileNetV2 do modelo V2. No ImageNet ela acerta 75,6%, contra 71,3% da V2. Na análise, num notebook sem GPU, ela leva ~29 ms por quadro a 224 px, contra ~21 ms do V2 a 160 px: segue acima de 30 quadros por segundo. A `EfficientNetV2B0` acerta mais (78,7%), mas é cerca de 2× mais lenta. A MobileNetV3 e a EfficientNetV2 já trazem a normalização dos pixels dentro delas. A MobileNetV2 precisa da camada `Rescaling`, que o código põe sozinho.

**Aumento de dados.** A cada época a rede vê versões diferentes de cada foto e aprende o que **não** importa para a resposta:

- espelhar, deslocar até 8%, girar até ±14° e aproximar ou afastar até 20%: o rosto não fica sempre no mesmo lugar nem na mesma distância da câmera;
- brilho e contraste ±25%: dia, noite, luz do painel;
- 15% das fotos em **cinza**: parecido com a câmera infravermelha;
- chuvisco de webcam: entra na leitura dos dados (seção 5).

No uso real (`predict`), todas essas camadas ficam desligadas.

**Por que `base(x, training=False)`?** A base tem camadas de *BatchNormalization*, que guardam médias aprendidas no ImageNet. Com `training=False`, elas continuam usando essas médias mesmo no ajuste fino. Sem isso, um lote pequeno bagunçaria as médias e estragaria a base.

In [ ]:
BASES = {   # nome: (construtor, a base já normaliza os pixels de 0 a 255?)
    "MobileNetV3Large": (keras.applications.MobileNetV3Large, True),
    "EfficientNetV2B0": (keras.applications.EfficientNetV2B0, True),
    "MobileNetV2": (keras.applications.MobileNetV2, False),
}
construtor, normaliza_sozinha = BASES[BASE]

camadas_aumento = [
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.08, 0.08, fill_mode="reflect"),
    layers.RandomRotation(0.04, fill_mode="reflect"),            # 0,04 × 360° = ±14,4°
    layers.RandomZoom((-0.2, 0.2), fill_mode="reflect"),
    layers.RandomBrightness(0.25, value_range=(0, 255)),
    layers.RandomContrast(0.25),
    layers.RandomGrayscale(0.15),
]
if AUMENTO_FORTE:   # cor e nitidez de outras câmeras e luzes, e partes do rosto escondidas
    camadas_aumento += [
        layers.RandomSaturation(0.3, value_range=(0, 255)),
        layers.RandomHue(0.04, value_range=(0, 255)),
        layers.RandomGaussianBlur(factor=0.3, sigma=(0.1, 1.5), value_range=(0, 255)),
        layers.RandomErasing(factor=0.25, scale=(0.02, 0.08), value_range=(0, 255)),
    ]
aumento = keras.Sequential(camadas_aumento, name="aumento_dados")

base = construtor(weights=PESOS_BASE, include_top=False, input_shape=(TAMANHO, TAMANHO, 3))
base.trainable = False

entrada = keras.Input((TAMANHO, TAMANHO, 3), name="imagem_motorista")
x = aumento(entrada)
if not normaliza_sozinha:
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)   # 0..255 → -1..1
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(DROPOUT)(x)
saida = layers.Dense(len(NOMES), activation="softmax", name="classificacao")(x)
modelo = keras.Model(entrada, saida, name=NOME_MODELO)

modelo.summary()

## 7. Treino, fase 1: só a camada nova aprende

- **Adam** com taxa de aprendizado 0,001: o algoritmo que ajusta os pesos.
- **Entropia cruzada com suavização de rótulos** (`SUAVIZACAO_ROTULOS = 0.1`): a "nota de erro" pune a rede quando ela dá probabilidade baixa à classe certa. Com a suavização, o alvo de uma foto sonolenta deixa de ser `[0, 1, 0]` e vira `[0,03, 0,93, 0,03]`. A rede para de buscar 100% de certeza, que é onde ela começa a decorar detalhes das fotos de treino. Assim ela erra menos em fotos novas, e as probabilidades ficam mais realistas, o que importa porque o projeto tira a **média** delas.
- **Pesos por classe** (`class_weight`): errar uma classe com poucas fotos custa mais. Assim a rede não ganha nota "chutando" a classe maior.
- **A melhor época é escolhida pela F1 macro da validação**, e não pelo erro. A F1 macro é a média do equilíbrio entre precision e recall **de cada classe**: uma rede que ignora o bocejo fica com nota baixa, mesmo que acerte muito o resto.
- **EarlyStopping**: se a F1 da validação não melhorar em algumas épocas, o treino para e volta aos pesos da melhor época.
- **ReduceLROnPlateau**: se o erro da validação empacar por 2 épocas, a taxa de aprendizado cai para 30%. Passos menores ajudam a achar um ponto melhor.
- **Uma linha por época** (`verbose=2`) aqui e em `resultados/treino_fase1.csv`, gravado a cada época.

In [ ]:
pesos = compute_class_weight("balanced", classes=np.arange(len(NOMES)), y=rotulos[idx_treino])
PESOS_CLASSE = dict(enumerate(pesos))
print("Pesos por classe:", {NOMES[i]: round(float(p), 2) for i, p in PESOS_CLASSE.items()})

PERDA = keras.losses.CategoricalCrossentropy(label_smoothing=SUAVIZACAO_ROTULOS)


def metricas():
    return ["accuracy", keras.metrics.F1Score(average="macro", name="f1")]


def callbacks(paciencia, fase):
    # treino_fase1.csv / treino_fase2.csv: uma linha por época, gravada na hora. Para acompanhar
    # um treino rodando fora do editor: tail -f treinamento/resultados/treino_fase1.csv
    return [keras.callbacks.EarlyStopping(monitor="val_f1", mode="max", patience=paciencia,
                                          restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2,
                                              min_lr=1e-7),
            keras.callbacks.CSVLogger(Path(PASTA_SAIDA) / f"treino_fase{fase}.csv")]


modelo.compile(optimizer=keras.optimizers.AdamW(1e-3, weight_decay=DECAIMENTO_PESOS), loss=PERDA,
               metrics=metricas())
inicio = time.time()
hist_fase1 = modelo.fit(treino, validation_data=validacao, epochs=EPOCAS, verbose=2,
                        class_weight=PESOS_CLASSE, callbacks=callbacks(paciencia=5, fase=1))
print(f"Fase 1: {(time.time() - inicio) / 60:.1f} min")
historicos = [hist_fase1]

### Curvas de aprendizado

Como ler:

- **as curvas de treino e de validação melhorando juntas**: está aprendendo;
- **treino continua melhorando, validação piora**: *overfitting* (decoreba). O EarlyStopping corta aqui;
- **validação melhor que treino**: normal, porque o aumento de dados, o Dropout e a suavização só agem no treino. Se a diferença for exagerada, desconfie de vazamento (seção 4).

In [ ]:
def mostrar_curvas(historicos, arquivo=None):
    juntar = lambda chave: sum((h.history[chave] for h in historicos), [])
    epocas = np.arange(1, len(juntar("loss")) + 1)
    fig, eixos = plt.subplots(1, 3, figsize=(16, 4))
    for eixo, chave, titulo in [(eixos[0], "loss", "Erro (loss)"),
                                (eixos[1], "accuracy", "Acurácia"),
                                (eixos[2], "f1", "F1 macro (escolhe a melhor época)")]:
        eixo.plot(epocas, juntar(chave), label="treino")
        eixo.plot(epocas, juntar("val_" + chave), label="validação")
        if len(historicos) > 1:
            eixo.axvline(len(historicos[0].history["loss"]) + 0.5, color="gray", linestyle="--",
                         label="início do ajuste fino")
        eixo.set_title(titulo)
        eixo.set_xlabel("época")
        eixo.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        eixo.legend()
    if arquivo:
        fig.savefig(arquivo, dpi=150, bbox_inches="tight")
    plt.show()


mostrar_curvas(historicos)

## 8. Treino, fase 2: ajuste fino (opcional)

Na fase 1 a base ficou congelada. Agora a parte final dela (`FRACAO_LIBERADA`, 20% das camadas) volta a aprender. São as camadas que reconhecem padrões mais específicos, como "olho meio fechado" ou "boca bem aberta". As primeiras (bordas, texturas) servem para qualquer foto e continuam congeladas.

- A taxa de aprendizado começa em 3e-5, 33 vezes menor que na fase 1. Se fosse alta, a rede apagaria em poucos passos o que aprendeu no ImageNet. O `compile` é obrigatório depois de mudar o que é treinável.
- Só vale a pena **depois** da fase 1. Se a camada nova ainda estiver aleatória, os erros grandes dela estragam a base.
- **Rede de segurança:** os pesos da fase 1 ficam guardados. Se, no fim, o ajuste fino tiver uma F1 de validação **pior**, o modelo volta para a fase 1. O ajuste fino nunca piora o resultado.

In [ ]:
if AJUSTE_FINO:
    guardado_fase1 = Path(PASTA_SAIDA) / "fase1.weights.h5"
    modelo.save_weights(guardado_fase1)
    f1_fase1 = modelo.evaluate(validacao, verbose=0, return_dict=True)["f1"]

    liberadas = int(len(base.layers) * FRACAO_LIBERADA)
    base.trainable = True
    for camada in base.layers[:-liberadas]:
        camada.trainable = False
    for camada in base.layers:
        if isinstance(camada, layers.BatchNormalization):
            camada.trainable = False   # as médias do ImageNet ficam como estão
    print(f"Camadas treináveis na base: {sum(c.trainable for c in base.layers)} de {len(base.layers)}")

    modelo.compile(optimizer=keras.optimizers.AdamW(3e-5, weight_decay=DECAIMENTO_PESOS), loss=PERDA,
                   metrics=metricas())
    inicio = time.time()
    hist_fase2 = modelo.fit(treino, validation_data=validacao, epochs=EPOCAS_AJUSTE_FINO, verbose=2,
                            class_weight=PESOS_CLASSE, callbacks=callbacks(paciencia=5, fase=2))
    print(f"Fase 2: {(time.time() - inicio) / 60:.1f} min")
    historicos.append(hist_fase2)

    f1_fase2 = modelo.evaluate(validacao, verbose=0, return_dict=True)["f1"]
    print(f"F1 macro na validação: fase 1 = {f1_fase1:.3f}, ajuste fino = {f1_fase2:.3f}")
    if f1_fase2 < f1_fase1:
        modelo.load_weights(guardado_fase1)
        print("O ajuste fino piorou: o modelo voltou para os pesos da fase 1.")
    guardado_fase1.unlink()

mostrar_curvas(historicos, Path(PASTA_SAIDA) / "curvas_treino.png")

## 9. Avaliação no teste (a nota honesta)

Agora sim o teste é usado, e só esta vez. Para cada imagem, a classe prevista é a de maior probabilidade (`argmax`).

- **precision** de sonolento: das vezes que a rede disse "sonolento", quantas estavam certas (muitos alarmes falsos = precision baixa);
- **recall** de sonolento: dos motoristas realmente sonolentos, quantos ela pegou. **É a métrica mais importante aqui**, porque deixar passar alguém dormindo é muito pior que um alarme falso;
- **f1**: equilíbrio entre as duas;
- **matriz de confusão**: linha = classe real, coluna = classe prevista. A diagonal são os acertos, e o resto mostra *qual* classe é confundida com *qual*.

Imagens e rótulos são lidos **no mesmo laço**, para a previsão de cada foto ficar junto do rótulo dela.

**Duas notas.** A primeira tabela tem o teste inteiro. As seguintes separam:

- **pessoas que a rede nunca viu** (os vídeos do RLDD do teste, já limpos pela seção 4.4): a nota que vale para um motorista novo. Hoje elas só têm alerta e sonolento;
- **fotos em blocos** (as pessoas das fotos próprias, que também estão no treino): otimista, mas é a única medida do bocejo enquanto ele tiver só 2 pessoas.

In [ ]:
if "teste" not in conjuntos:
    print("Sem teste separado (TESTE_SEPARADO = False): a avaliação abaixo usa a validação, que "
          "escolheu a melhor época, e fica otimista. A nota com gente nova é a da validação cruzada.\n")
y_real, y_prob = [], []
for imagens, rots in teste:
    y_prob.append(modelo.predict_on_batch(imagens))
    y_real.append(rots.numpy().argmax(axis=1))
y_real, y_prob = np.concatenate(y_real), np.concatenate(y_prob)
y_prev = y_prob.argmax(axis=1)



def relatorio_de(mascara, titulo):
    """Imprime o relatório das imagens em `mascara` e o devolve em dicionário (ou None)."""
    if not mascara.any():
        return None
    presentes = sorted(set(y_real[mascara]))
    print(f"--- {titulo}: {mascara.sum()} imagens ---")
    print(classification_report(y_real[mascara], y_prev[mascara], labels=presentes,
                                target_names=[NOMES[i] for i in presentes], digits=3, zero_division=0))
    rel = classification_report(y_real[mascara], y_prev[mascara], labels=presentes,
                                target_names=[NOMES[i] for i in presentes], digits=3,
                                zero_division=0, output_dict=True)
    return {"imagens": int(mascara.sum()),
            "acuracia": round(float((y_real[mascara] == y_prev[mascara]).mean()), 4),
            "f1_macro": round(rel["macro avg"]["f1-score"], 4),
            "recall": {NOMES[i]: round(rel[NOMES[i]]["recall"], 4) for i in presentes},
            "precision": {NOMES[i]: round(rel[NOMES[i]]["precision"], 4) for i in presentes}}


relatorio = classification_report(y_real, y_prev, labels=range(len(NOMES)), target_names=NOMES,
                                  digits=3, zero_division=0, output_dict=True)
print(classification_report(y_real, y_prev, labels=range(len(NOMES)), target_names=NOMES,
                            digits=3, zero_division=0))
# `teste` segue a ordem de idx_teste (sem embaralhar); se alguma imagem falhou, não dá para separar
notas_separadas = {}
if len(y_real) == len(idx_teste) and (y_real == rotulos[idx_teste]).all():
    notas_separadas = {
        "pessoas_novas": relatorio_de(~em_bloco[idx_teste], "pessoas que a rede nunca viu"),
        "fotos_em_blocos": relatorio_de(em_bloco[idx_teste], "fotos em blocos (otimista)"),
    }
else:
    print("Alguma imagem do teste não foi lida: as notas separadas ficaram de fora.")

# Nota das pessoas novas com o alarme a partir de P(sonolento) ≥ 30%, e tudo num arquivo, para a
# validação cruzada juntar as dobras.
novas = ~em_bloco[idx_teste] if len(y_real) == len(idx_teste) else np.zeros(len(y_real), bool)
SONO = NOMES.index("sonolento")
alarme_30 = {
    "recall_sonolento": round(float(np.mean(y_prob[novas & (y_real == SONO), SONO] >= 0.3)), 4)
    if (novas & (y_real == SONO)).any() else None,
    "falso_alarme_alerta": round(float(np.mean(y_prob[novas & (y_real == 0), SONO] >= 0.3)), 4)
    if (novas & (y_real == 0)).any() else None,
}
por_pessoa = {}
for quem in sorted(set(pessoas[idx_teste][novas])):
    dela = novas & (pessoas[idx_teste] == quem)
    por_pessoa[quem] = {NOMES[c]: round(float(np.mean(y_prev[dela & (y_real == c)] == c)), 4)
                        for c in sorted(set(y_real[dela]))}
(Path(PASTA_SAIDA) / "metricas_teste.json").write_text(json.dumps({
    "modelo": NOME_MODELO, "dobra": DOBRA, "dobras": DOBRAS if DOBRA is not None else None,
    "acuracia": round(relatorio["accuracy"], 4), "f1_macro": round(relatorio["macro avg"]["f1-score"], 4),
    **notas_separadas, "alarme_30_pessoas_novas": alarme_30, "por_pessoa": por_pessoa,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Pessoas novas, alarme com P(sonolento) ≥ 30%:", alarme_30)

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
matriz = confusion_matrix(y_real, y_prev, labels=range(len(NOMES)))
ConfusionMatrixDisplay(matriz, display_labels=NOMES).plot(ax=eixos[0], cmap="Blues",
                                                          values_format="d", colorbar=False)
eixos[0].set_title("Quantidade")
ConfusionMatrixDisplay(confusion_matrix(y_real, y_prev, labels=range(len(NOMES)), normalize="true"),
                       display_labels=NOMES).plot(ax=eixos[1], cmap="Blues", values_format=".0%",
                                                  colorbar=False)
eixos[1].set_title("Por linha (recall de cada classe)")
for eixo in eixos:
    eixo.set_xlabel("previsto")
    eixo.set_ylabel("real")
fig.suptitle(f"Matriz de confusão no teste - {NOME_MODELO} ({BASE}, {TAMANHO} px)")
fig.savefig(Path(PASTA_SAIDA) / "matriz_confusao.png", dpi=150, bbox_inches="tight")
plt.show()

### E se o alarme disparar antes do 50%?

Com o `argmax`, a rede só diz "sonolento" quando essa é a maior probabilidade. Num sistema de segurança, pode valer a pena alertar mais cedo, por exemplo com P(sonolento) ≥ 30%: pega mais motoristas sonolentos (recall ↑) ao custo de mais alarmes falsos (precision ↓). A tabela mostra essa troca no teste e ajuda a escolher o limiar do projeto.

In [ ]:
SONOLENTO = NOMES.index("sonolento")
real = y_real == SONOLENTO
print(f"{'limiar':>7} {'recall':>8} {'precision':>10} {'alarmes':>8}")
for limiar in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    alarme = y_prob[:, SONOLENTO] >= limiar
    recall = (alarme & real).sum() / max(real.sum(), 1)
    precisao = (alarme & real).sum() / max(alarme.sum(), 1)
    print(f"{limiar:>7.0%} {recall:>8.1%} {precisao:>10.1%} {alarme.sum():>8}")

## 10. Salvar o modelo na pasta `modelo/`

São salvos dois arquivos em `modelo/`, a pasta que o projeto usa (`classificador.modelo` em `config/analise.json`):

- `cnn_sonolencia.keras`: arquitetura, pesos e pré-processamento, tudo junto;
- `cnn_sonolencia.classes.json`: a ordem das saídas, a base, o tamanho de entrada e as notas no teste. O projeto lê este arquivo e confere com `classificador.classes`.

Para não perder nada:

1. o modelo novo é salvo primeiro com outro nome e **recarregado**, e a célula confere se ele prevê exatamente o mesmo que o da memória;
2. só então o modelo anterior (se houver) vai para `modelo/anteriores/<data e hora>/`;
3. e o novo assume o nome `cnn_sonolencia.keras`.

Para voltar ao modelo anterior, é só mover os dois arquivos de `anteriores/` de volta para `modelo/`.

In [ ]:
if not SALVAR_MODELO:
    print("SALVAR_MODELO = False (validação cruzada): o modelo desta dobra não vai "
          "para modelo/. As notas ficaram em", Path(PASTA_SAIDA) / "metricas_teste.json")
else:
    pasta_modelo = Path(PASTA_MODELO)
    caminho_modelo = pasta_modelo / f"{ARQUIVO_MODELO}.keras"
    caminho_classes = pasta_modelo / f"{ARQUIVO_MODELO}.classes.json"
    provisorio = pasta_modelo / f"{ARQUIVO_MODELO}.novo.keras"

    modelo.save(provisorio)
    recarregado = keras.models.load_model(provisorio, compile=False)
    lote = next(iter(teste))[0]
    np.testing.assert_allclose(recarregado.predict_on_batch(lote), modelo.predict_on_batch(lote),
                               atol=1e-5)

    anteriores = [c for c in (caminho_modelo, caminho_classes) if c.exists()]
    if anteriores:
        guarda = pasta_modelo / "anteriores" / time.strftime("%Y-%m-%d_%H%M%S")
        guarda.mkdir(parents=True)
        for arquivo in anteriores:
            shutil.move(arquivo, guarda / arquivo.name)
        print("Modelo anterior guardado em", guarda)
    provisorio.rename(caminho_modelo)

    caminho_classes.write_text(json.dumps({
        "modelo": NOME_MODELO,
        "classes": NOMES,
        "pastas": CLASSES,
        "base": BASE,
        "entrada": [TAMANHO, TAMANHO, 3],
        "recorte": "rosto" if RECORTAR_ROSTO else "quadro",   # lido por entrada/classificador.py
        "fator_recorte": FATOR_RECORTE if RECORTAR_ROSTO else None,
        "pixels": ("RGB de 0 a 255, quadrado do rosto (caixa_do_rosto) redimensionado" if RECORTAR_ROSTO
                   else "RGB de 0 a 255, quadro inteiro redimensionado sem recorte"),
        "divisao": "por pessoa" if REGEX_PESSOA else "por vídeo/foto",
        "pessoas_em_blocos": pessoas_em_blocos,
    "teste_separado": TESTE_SEPARADO,
    "validacao_cruzada": (json.loads(Path(RESUMO_VALIDACAO_CRUZADA).read_text(encoding="utf-8"))
                          if RESUMO_VALIDACAO_CRUZADA else None),
    "contra_decoreba": {"max_por_pessoa_classe": MAX_POR_PESSOA_CLASSE, "fracao_liberada": FRACAO_LIBERADA,
                        "dropout": DROPOUT, "decaimento_pesos": DECAIMENTO_PESOS,
                        "aumento_forte": AUMENTO_FORTE},   # em blocos de fotos em todas as classes, não por pessoa
        "quadros_dos_videos": {n: config_quadros(n) for n in NOMES if videos[n]},
        "limpeza_dos_rotulos": {"fracao_olho": FRACAO_OLHO} if LIMPAR_ROTULOS else None,
        "exemplos": {"fotos": int((~de_video).sum()), "quadros_de_video": int(de_video.sum())},
        "teste": {
            "imagens": int(len(y_real)),
            "acuracia": round(relatorio["accuracy"], 4),
            "f1_macro": round(relatorio["macro avg"]["f1-score"], 4),
            "recall": {n: round(relatorio[n]["recall"], 4) for n in NOMES},
            "precision": {n: round(relatorio[n]["precision"], 4) for n in NOMES},
            **notas_separadas,
        },
        "treinado_em": date.today().isoformat(),
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    print("Salvo e conferido:")
    print(" ", caminho_modelo)
    print(" ", caminho_classes)

## 11. Usar no projeto

Pronto. A análise (`python -m entrada`) e o simulador já leem `modelo/cnn_sonolencia.keras`. O tamanho de entrada vem do próprio modelo, e o recorte (`"recorte": "rosto"`) vem do `.classes.json`: com ele, a análise passa os pontos do rosto ao classificador e a página do simulador recorta o rosto antes de mandar o quadro.

Se `classificador.classes` em `config/analise.json` não for `["alerta", "sonolento", "bocejo"]`, o projeto avisa a diferença e não carrega o modelo.

**O que mais aumentaria o acerto:**

- **Dados parecidos com o uso.** Vídeos gravados **na mesma webcam e posição** em que o sistema vai rodar, de **várias pessoas**, com e sem óculos, em luzes diferentes. Nenhuma técnica compensa a diferença entre fotos de treino e imagens de uso: é por isso que o V2 dava "sonolento" para um rosto acordado.
- **O que esperar do RLDD.** O rótulo de cada vídeo vale para ele inteiro, mas uma pessoa "sonolenta" passa boa parte do vídeo de olhos abertos. Num quadro isolado, separar alerta de sonolento em gente nova é difícil: entre 65% e 75% de acerto já seria um bom resultado. A média de 2 s da análise (`janela_ms`) ajuda no uso.
- **Bocejos de mais pessoas.** O bocejo só tem fotos de 2 pessoas (o RLDD não tem essa classe). Com a divisão por pessoa, uma delas pode ir inteira para o teste e a rede aprende bocejo com uma pessoa só. Vídeos curtos de bocejo de várias pessoas, ou o dataset público YawDD (pedido aos autores), são o que mais falta.